In [1]:
# PHASE 2: Fetching Sentinel-1 SAR Radar Data (Sundarbans & Cyclone Remal)

# 1. Load tools and turn on the engine (Required for EVERY new notebook)
import ee
import geemap
ee.Initialize()

# 2. Define the exact boundary for the Sundarbans analysis area
sundarbans_aoi = ee.Geometry.Rectangle([88.0, 21.5, 90.5, 22.6])

# 3. Define the exact dates for our Case Study: Cyclone Remal (May 2024)
pre_flood_start = '2024-04-01'
pre_flood_end = '2024-05-15'
peak_flood_start = '2024-05-26'
peak_flood_end = '2024-06-15'

# 4. Connect to the Sentinel-1 Satellite Database (Filtering for 'VH' water detection)
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(sundarbans_aoi) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .select('VH')

# 5. Separate the data into our "Before" and "After" folders
pre_flood_data = s1_collection.filterDate(pre_flood_start, pre_flood_end)
peak_flood_data = s1_collection.filterDate(peak_flood_start, peak_flood_end)

# 6. Ask Google's servers to count how many satellite images we captured
print("Environment Initialized. Satellite Data Successfully Locked In!")
print(f"Pre-Flood Images Found: {pre_flood_data.size().getInfo()}")
print(f"Peak-Flood Images Found: {peak_flood_data.size().getInfo()}")

Environment Initialized. Satellite Data Successfully Locked In!
Pre-Flood Images Found: 23
Peak-Flood Images Found: 14


In [3]:
#1. SQUASH AND CLIP: Turn the stacks into single, clean images
pre_flood_img = pre_flood_data.median().clip(sundarbans_aoi)
peak_flood_img = peak_flood_data.median().clip(sundarbans_aoi)

#2. Create a fresh map canvas
Map2 = geemap.Map()
Map2.setCenter(89.2000, 21.9000, 8)

#3. Define Visualization Rules for Radar
# Radar data is measured in decibles (dB).
# -25 dB is very dark (smooth water), 0 dB is very bright (rough land/buildings)
vis_params = {'min': -25, 'max' : 0}

#4. Add our processed images to the map as distinct layers
Map2.addLayer(pre_flood_img, vis_params, 'Pre-Flood Baseline (April/May)')
Map2.addLayer(peak_flood_img, vis_params, 'Peak-Flood (Cyclone Remal)')

#5. Display the map
Map2

NameError: name 'pre_flood_data' is not defined

In [12]:
# 1. THRESHOLDING: Define what "Water" looks like to the radar
# Radar values below -18 dB are generally smooth surfaces (like water)
water_threshold = -18

# Find all pixels that are water (Value becomes 1 for water, 0 for land)
pre_water = pre_flood_img.lt(water_threshold)
peak_water = peak_flood_img.lt(water_threshold)

# 2. CHANGE DETECTION: Isolate the actual floodwaters
# We subtract the 'Before' water from the 'After' water. 
# What is left over is the NEW water (The Flood!)
flood_extent = peak_water.subtract(pre_water).gt(0)

# 3. MASKING: Make the non-water areas transparent so we can see the base map
pre_water_masked = pre_water.updateMask(pre_water)
flood_extent_masked = flood_extent.updateMask(flood_extent)

# 4. VISUALIZATION: Create a new map and apply our professional color palette
Map3 = geemap.Map()
Map3.setCenter(89.2000, 21.9000, 8)

# Add the base grayscale radar image for geographic context (set to 50% opacity)
Map3.addLayer(pre_flood_img, {'min': -25, 'max': 0}, 'Base Landscape', True, 0.5)

# Add the Permanent Water in Deep Blue
Map3.addLayer(pre_water_masked, {'palette': ['0000FF']}, 'Permanent Water', True)

# Add the Flood Extent in Bright Red (Standard for disaster mapping to make it pop)
Map3.addLayer(flood_extent_masked, {'palette': ['FF0000']}, 'Flood Extent', True)

# 5. ADD THE LEGEND
legend_colors = {
    'Permanent Water': '0000FF',
    'Flood Extent (Cyclone Remal)': 'FF0000'
}
Map3.add_legend(title="Flood Impact Analysis", legend_dict=legend_colors)

# Display the final map
Map3

Map(center=[21.9, 89.2], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …